# Train and visualize (catalog continuation prototype)

Set env `CATALOG_TFM_DATA_DIR` to your ingested folder, or rely on `../eq_mag_prediction/results/catalogs/ingested` from cwd. If that folder has no CSVs, the notebook uses `scripts/fixtures/minimal_catalog.csv`.


In [ ]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import pandas as pd

from eq_mag_prediction.utilities import catalog_processing
from eq_mag_prediction.utilities import loading_utils

from catalog_tfm import data
from catalog_tfm.model import build_model

REPO_ROOT = Path.cwd()
DEFAULT_INGESTED = REPO_ROOT.parent / "eq_mag_prediction" / "results" / "catalogs" / "ingested"
FIXTURE_DIR = REPO_ROOT / "scripts" / "fixtures"

env_dir = os.environ.get("CATALOG_TFM_DATA_DIR")
DATA_DIR = Path(env_dir) if env_dir else DEFAULT_INGESTED
if (not DATA_DIR.is_dir()) or (not list(DATA_DIR.glob("*.csv"))):
    DATA_DIR = FIXTURE_DIR

resolved = data.resolve_data_dir(DATA_DIR)
print("data_dir:", resolved)
print("eq_mag get_resource_path(results/catalogs/ingested):", loading_utils.get_resource_path("results/catalogs/ingested"))


In [ ]:
seq_len = 16
X, y, hashes = data.load_all_windows(resolved, seq_len)
print("X, y:", X.shape, y.shape)
for k in sorted(hashes):
    print("catalog_sha256", k, hashes[k])

first_csv = sorted(resolved.glob("*.csv"))[0]
sample = pd.read_csv(first_csv)
print("hash(raw read):", catalog_processing.hash_pandas_object(sample))


In [ ]:
import tensorflow as tf
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)
scaler = data.fit_scaler(X_train)
X_train_t = data.transform_X(X_train, scaler)
X_val_t = data.transform_X(X_val, scaler)

tf.random.set_seed(0)
model = build_model(seq_len, X.shape[-1], d_model=32, num_heads=4, ff_dim=64, num_layers=1)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mse", metrics=["mae"])
hist = model.fit(
    X_train_t,
    y_train,
    validation_data=(X_val_t, y_val),
    epochs=5,
    batch_size=32,
    verbose=1,
)


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(hist.history["loss"], label="train")
ax[0].plot(hist.history["val_loss"], label="val")
ax[0].set_title("MSE loss")
ax[0].legend()
ax[1].plot(hist.history["mae"], label="train")
ax[1].plot(hist.history["val_mae"], label="val")
ax[1].set_title("MAE")
ax[1].legend()
plt.tight_layout()
plt.show()


In [ ]:
pred = model.predict(X_val_t[:200], verbose=0).ravel()
plt.figure(figsize=(5, 5))
plt.scatter(y_val[:200], pred, s=8, alpha=0.6)
m = float(min(y_val[:200].min(), pred.min()))
M = float(max(y_val[:200].max(), pred.max()))
plt.plot([m, M], [m, M], "k--", lw=1)
plt.xlabel("actual magnitude")
plt.ylabel("predicted")
plt.title("Next-event magnitude (holdout sample)")
plt.show()
